# The largest stable step for a cubic fit

A thermistor's response is measured at twenty settings, scaled to run from $-1$ to $1$, and a
cubic $\hat{y} = \theta_0 + \theta_1 t + \theta_2 t^2 + \theta_3 t^3$ is to be fitted to the readings
by gradient descent on the mean squared error. The model is linear in $\boldsymbol\theta$, so
the loss is a quadratic bowl, and its Hessian decides how large a learning rate can be before
the steps start to grow instead of shrink. You build the features, find that limit from the
Hessian, and show that gradient descent converges just below it and blows up just above it.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Twenty readings, settings scaled to [-1, 1].
rng = np.random.default_rng(7)
t = np.linspace(-1, 1, 20)
y = 0.5 - 1.2 * t + 0.8 * t**3 + 0.05 * rng.standard_normal(20)
print(np.round(y, 3))

## Part 1 (coding task): The features

Write `poly_features(t, degree)`, which takes a 1-D array of $N$ inputs and returns an
$N \times (\text{degree} + 1)$ array whose column $j$ is $t^j$: the constant column first, then
$t$, $t^2$ and so on. Use no Python loop.

In [ ]:
def poly_features(t, degree):
    """Column j holds t**j, for j = 0, ..., degree."""
    # TODO
    raise NotImplementedError

In [ ]:
P = poly_features(np.array([2.0, -1.0, 0.5]), 3)
assert P.shape == (3, 4), f"expected shape (3, 4), got {P.shape}"
assert not np.allclose(P, np.vander([2.0, -1.0, 0.5], 4)), "the powers run the wrong way: column 0 must be the constant column, column j must hold t**j"
assert np.allclose(P, [[1, 2, 4, 8], [1, -1, 1, -1], [1, 0.5, 0.25, 0.125]]), f"some entries are wrong:\n{P}"
P0 = poly_features(t, 0)
assert P0.shape == (20, 1) and np.allclose(P0, 1), "degree 0 should give the constant column alone, shape (N, 1)"
print("Part 1 passed")

## Part 2 (coding task): The largest stable learning rate

The loss is $\mathcal{L}(\boldsymbol\theta) = \tfrac{1}{N}\,\lVert\mathbf{y} - X\boldsymbol\theta\rVert^2$. Its Hessian
$H$ is the same at every $\boldsymbol\theta$, and its gradient vanishes at the least-squares fit
$\boldsymbol\theta^\star$, so the gradient at any $\boldsymbol\theta$ is exactly $H(\boldsymbol\theta - \boldsymbol\theta^\star)$.

Work out what a single update of gradient descent, at rate $\eta$, does to the error
$\boldsymbol\theta - \boldsymbol\theta^\star$ along an eigenvector of $H$ with eigenvalue $\lambda$. Then write
`stable_rate(X)`, which returns the largest $\eta$ for which the error does not grow along any
eigenvector (the boundary value itself). You may use `np.linalg`.

In [ ]:
def stable_rate(X):
    """The learning rate at which gradient descent on the mean squared error stops converging."""
    # TODO
    raise NotImplementedError

In [ ]:
X3 = poly_features(t, 3)
rate = float(stable_rate(X3))
lams = np.linalg.eigvalsh(X3.transpose().dot(X3) * (2 / len(t)))
assert not np.isclose(rate, 1 / lams.max()), "that is the rate that removes the steepest direction's error in one step, but the error only grows once the factor 1 - rate * lambda drops below -1"
assert not np.isclose(rate, 2 / lams.min()), "use the largest eigenvalue: the steepest direction is the first to overshoot"
assert not np.isclose(rate, 2 / np.linalg.eigvalsh(X3.transpose().dot(X3)).max()), "the Hessian of the mean squared error is (2/N) X^T X, not X^T X"
assert np.isclose(rate, 2 / lams.max()), f"stable_rate gave {rate:.4f}, expected {2 / lams.max():.4f}"
R = np.random.default_rng(1).standard_normal((7, 3))
assert np.isclose(stable_rate(R), 2 / np.linalg.eigvalsh(R.transpose().dot(R) * (2 / 7)).max()), "wrong on a different X: compute the rate from the X you are given"
print(f"largest stable rate: {rate:.4f}")
print("Part 2 passed")

## Part 3 (coding task): Below the limit and above it

Write `fit(X, y, lr, steps)`. It starts from $\boldsymbol\theta = \mathbf{0}$, takes `steps` gradient
descent steps on the mean squared error with learning rate `lr`, and returns two things: the
final $\boldsymbol\theta$, and an array `losses` of length `steps + 1` whose entry $k$ is the loss after
$k$ steps (so `losses[0]` is the loss at the start). A loop over the steps is fine. The check
runs it at 95% and at 105% of your `stable_rate`.

In [ ]:
def fit(X, y, lr, steps):
    """Gradient descent from zero. Returns (theta, losses) with len(losses) == steps + 1."""
    # TODO
    raise NotImplementedError

In [ ]:
X3 = poly_features(t, 3)
rate = float(stable_rate(X3))
theta_gd, losses = fit(X3, y, lr=0.95 * rate, steps=2000)
losses = np.asarray(losses, dtype=float)
assert losses.shape == (2001,), f"losses should hold the start and every step, 2001 entries, got shape {losses.shape}"
assert np.isclose(losses[0], np.mean(y ** 2)), "losses[0] should be the loss at theta = 0, before any step"
assert losses[1] < losses[0], "the first step raised the loss: step against the gradient, theta - lr * grad"
assert np.all(np.diff(losses) <= 1e-12), "below the limit the loss should never go up"
best = np.linalg.lstsq(X3, y, rcond=None)[0]
assert np.allclose(theta_gd, best, atol=1e-6), f"after 2000 steps theta should match the least-squares fit {np.round(best, 4)}, got {np.round(theta_gd, 4)}"
_, over = fit(X3, y, lr=1.05 * rate, steps=300)
assert over[-1] > 1e6 * over[0], f"at 105% of the limit the loss should blow up, but after 300 steps it is {over[-1]:.3g}"
print(f"95% of the limit: fitted cubic {np.round(theta_gd, 3)}, final loss {losses[-1]:.5f}")
print(f"105% of the limit: loss {over[0]:.3f} at the start, {over[-1]:.3g} after 300 steps")
print("Part 3 passed")

## Solution

### Part 1: The features

In [ ]:
def poly_features(t, degree):
    """Column j holds t**j, for j = 0, ..., degree."""
    return np.asarray(t, dtype=float)[:, None] ** np.arange(degree + 1)

**Why it works.** A column of inputs, `t[:, None]`, raised to a row of exponents, `np.arange(degree + 1)`,
broadcasts to the whole table of $t_n^j$ in one operation. `np.vander` builds the same table
with the powers in decreasing order unless it is given `increasing=True`, which is the mistake
the check looks for: the fit would still work, but every coefficient would sit in the wrong place.

### Part 2: The largest stable learning rate

In [ ]:
def stable_rate(X):
    """The learning rate at which gradient descent on the mean squared error stops converging."""
    hessian = 2 / X.shape[0] * (X.T @ X)
    return 2 / np.linalg.eigvalsh(hessian).max()

**Why it works.** With $\mathbf{e} = \boldsymbol\theta - \boldsymbol\theta^\star$, one step gives
$\mathbf{e} \leftarrow \mathbf{e} - \eta H\mathbf{e} = (I - \eta H)\,\mathbf{e}$. Along an eigenvector with
eigenvalue $\lambda$ that is multiplication by $1 - \eta\lambda$, every step. The error shrinks
in that direction while $|1 - \eta\lambda| < 1$, that is $0 < \eta < 2/\lambda$, and the tightest of
these conditions comes from the largest eigenvalue: $\eta < 2/\lambda_{\max}$. Here the Hessian
$\tfrac2N X^\top X$ has eigenvalues from about $0.041$ to $2.30$, so the limit is about $0.870$.
The rate $1/\lambda_{\max}$ is a different landmark: it wipes out the steepest direction's
error in a single step.

### Part 3: Below the limit and above it

In [ ]:
def fit(X, y, lr, steps):
    """Gradient descent from zero. Returns (theta, losses) with len(losses) == steps + 1."""
    n = len(y)
    theta = np.zeros(X.shape[1])
    losses = [np.mean((y - X @ theta) ** 2)]
    for _ in range(steps):
        grad = -2 / n * (X.T @ (y - X @ theta))
        theta = theta - lr * grad
        losses.append(np.mean((y - X @ theta) ** 2))
    return theta, np.array(losses)

**Why it works.** Each step multiplies the error along the steepest eigenvector by $1 - \eta\lambda_{\max}$. At 95% of
the limit that factor is $1 - 1.9 = -0.9$: the error flips sign every step but shrinks, so the
loss still falls at every step. At 105% it is $-1.1$, and 300 steps multiply the error by
$1.1^{300} \approx 2.6 \times 10^{12}$. The other directions are slower: the flattest one shrinks by only
$1 - 0.95 \times 0.870 \times 0.041 \approx 0.966$ per step, which is why it takes a few hundred steps to
reach the fit. The ratio $\lambda_{\max}/\lambda_{\min} \approx 56$ sets that crawl. On the raw settings,
spread over $[0, 20]$ instead of $[-1, 1]$, the same cubic has a Hessian whose eigenvalues run from
about $0.16$ to $2 \times 10^7$: the limit drops to about $10^{-7}$ and the flattest direction needs on
the order of a billion steps. Scaling the inputs first is what makes gradient descent usable here.